# 03 JPL Horizons EDA

Request or reuse exact-match Sun/Earth responses for catalogued points. The configured UTC epoch is provisional.

In [ ]:
from pathlib import Path
import os, sys
root=Path(os.environ.get("LUNARSIGHT_ROOT", Path.cwd())).resolve()
if not (root/"src/pipeline.py").exists() and (root.parent/"src/pipeline.py").exists():
    root=root.parent
assert (root/"src/pipeline.py").exists(), "Set LUNARSIGHT_ROOT to the extracted lunarsight-data folder."
sys.path.insert(0,str(root/"src"))
for name in ("reports","manifests","processed","figures"): (root/name).mkdir(exist_ok=True)

In [ ]:
from acquire import load, inventory
from horizons import fetch
from pipeline import validate_config, plot_ephem
from IPython.display import Image, display
import pandas as pd
s=load(root/"config/scenario.json"); g=load(root/"config/geometry_config.json")
a,b,end=validate_config(s,g)
reports=[]
for f in load(root/"processed/site_catalog.geojson")["features"]:
    site=f["properties"]; series=[]
    for target,h in ((10,1),(399,2)):
        data,m=fetch(root,site,target,h,a,end,5,g["R_ref_m"],g["horizons_candidate_radius_m"])
        series.append(data);reports.append(m)
    assert series[0].index.equals(series[1].index)
    assert all(not r["missing_timestamps"] and not r["missing_angles"] for r in reports)
    plot_ephem(root,site["site_id"],*series)
    display(Image(filename=str(root/"figures"/f"{site['site_id']}_horizons.png")))
old=load(root/"manifests/source_manifest.json")
by_name={x["name"]:x for x in old.get("horizons",[])+reports}
inventory(root,old["records"],by_name.values())
display(pd.DataFrame(reports)[["target","step_minutes","received_rows","missing_timestamps","missing_angles","observer_frame"]])